# Clase 3 — Datos multimodales y embeddings

## Pregunta central

> **¿Cómo puede el mismo software trabajar con tablas, texto, imágenes, audio y datos satelitales?**

## Idea principal

Cada modalidad necesita una representación numérica y un preprocesamiento coherente; un embedding resume contenido en un vector.

## Objetivos de aprendizaje

Al finalizar la clase deberías poder:

- Reconocer la forma numérica de cinco modalidades.
- Distinguir datos crudos, features y embeddings.
- Interpretar dimensión y similitud coseno.
- Construir una búsqueda semántica pequeña.
- Comparar imágenes y representaciones de audio sin mezclar espacios.

## Recorrido de la clase

| Paso | Tema |
|---:|---|
| 1 | De datos reales a tensores |
| 2 | Features frente a embeddings |
| 3 | Similitud coseno |
| 4 | Búsqueda semántica |
| 5 | Embeddings de imagen y audio |

## Cómo trabajar con este notebook

1. Ejecutá las celdas en el orden propuesto.
2. Antes de modificar código, observá y describí el resultado.
3. Cambiá solamente las variables marcadas con `TODO`.
4. No es necesario implementar algoritmos desde cero.
5. Si aparece un término nuevo, buscá primero su definición en el glosario de la clase.

**Conexión con el programa:** embeddings y recuperación del Track Salud, y representaciones visuales/multibanda del Track Imagen.

## Glosario mínimo

| Término | Explicación breve |
|---|---|
| Modalidad | Tipo de dato: tabla, texto, imagen, audio o ráster |
| Representación | Forma numérica entregada al software |
| Preprocesamiento | Transformación previa consistente |
| Feature | Propiedad medida o calculada |
| Embedding | Vector producido por un modelo |
| Dimensión | Cantidad de valores del vector |
| Similitud coseno | Comparación de dirección entre vectores |
| Encoder | Modelo que produce una representación |
| Retrieval | Recuperación de elementos relevantes |
| Espacio compartido | Representaciones entrenadas para ser comparables |
| Tensor | Arreglo numérico con una forma y un tipo de dato |
| Batch | Grupo de muestras procesado en una misma operación |
| Token | Fragmento en el que un tokenizador divide un texto |
| Waveform | Secuencia de amplitudes de una señal a lo largo del tiempo |
| Sample rate | Cantidad de muestras de audio por segundo |
| ASR | Reconocimiento automático de voz: transforma habla en texto |
| CRS | Sistema que da significado geográfico a unas coordenadas |
| NDVI | Índice calculado con bandas roja e infrarroja para describir contraste espectral de vegetación |
| Head | Capa final que convierte features en la salida de una tarea |
| Producto punto | Multiplicar posiciones equivalentes de dos vectores y sumar los resultados |

---
## 1. La forma de cada modalidad

Una **modalidad** es una clase de información con estructura propia.
Para una persona, una frase y una imagen son contenidos distintos.
Para una computadora, ambos deben convertirse en números antes de
llegar a un modelo.

> **Analogía del cajón de herramientas.** Cada modalidad es una
> herramienta distinta: un martillo, una llave y un destornillador
> no se usan igual. La computadora tampoco puede tratar un audio
> como si fuera una tabla. Primero hay que "traducir" cada cosa a
> números, y esa traducción debe respetar la forma de cada una.

```text
fenómeno real
    |
    v
archivo o registro
    |
    v
lectura y preprocesamiento
    |
    v
tensor numérico
    |
    v
modelo
```

La conversión debe conservar la estructura útil de cada modalidad.

| Modalidad | Representación inicial | Estructura que importa | Ejemplo de forma |
|---|---|---|---|
| Tabular | Filas y columnas | Qué significa cada columna | `(muestras, features)` |
| Texto | Secuencia de tokens | Orden y contexto | `(tokens,)` |
| Imagen RGB | Intensidades por canal | Vecindad espacial | `(alto, ancho, 3)` |
| Audio | Amplitud en el tiempo | Orden temporal y sample rate | `(muestras_audio,)` |
| Ráster multibanda | Una grilla por banda | Posición, resolución y CRS | `(bandas, alto, ancho)` |

**La forma no es un detalle:** define qué operaciones son válidas.
Una banda espectral no es una fila tabular; un token no es un píxel.

> **Pensalo así:** la *shape* es como el molde de una galletita.
> Si el molde es redondo, no podés hacer galletitas cuadradas.
> La forma del tensor dice qué operaciones tienen sentido y cuáles no.

### Shape, dtype y rango

Un tensor se describe al menos con:

| Propiedad | Pregunta |
|---|---|
| Shape | ¿Cuántos ejes hay y cuánto mide cada uno? |
| Dtype | ¿Son enteros, decimales o booleanos? |
| Rango | ¿Los valores están entre 0–255, 0–1 u otra escala? |
| Semántica | ¿Qué representa cada eje y cada valor? |

Dos tensores pueden tener la misma shape y representar cosas
completamente distintas. `(4, 32, 32)` podría ser un ráster de
cuatro bandas o un batch de cuatro imágenes en escala de grises.

### Qué agrega un batch

Los modelos suelen procesar varias muestras juntas:

```text
una imagen RGB       -> (3, alto, ancho)
batch de 16 imágenes -> (16, 3, alto, ancho)
```

El primer eje pasa a indicar cuántas muestras contiene el lote.

> **Analogía del lote:** es como preparar 16 galletitas a la vez en
> una misma bandeja en lugar de una por una. El modelo ve un grupo
> completo, y el primer número de la shape cuenta cuántas hay.

In [ ]:
# --- Preparación: imports y datos de ejemplo ---
# Cargamos las librerías que usaremos en toda la clase.
import re
import ssl

import certifi
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_sample_image

# Semilla fija: los números aleatorios salen iguales en cada ejecución.
FAST_MODE = True
SEED = 42
rng = np.random.default_rng(SEED)
# Certificados SSL para descargar imágenes de ejemplo sin errores.
ssl._create_default_https_context = (
    lambda: ssl.create_default_context(cafile=certifi.where())
)

# Una muestra de cada modalidad: tabla, texto, imagen, audio y ráster.
tabla = pd.DataFrame({
    "edad": [34, 58, 41],
    "consultas_previas": [1, 6, 2],
})
texto = "El lote presenta vegetación saludable."
# Dividimos el texto en tokens (palabras y signos) para ver su forma.
tokens_didacticos = re.findall(
    r"\w+|[^\w\s]", texto.lower(), flags=re.UNICODE
)
imagen = load_sample_image("flower.jpg")
# Un tono de 220 Hz durante 1 segundo a 16 kHz (16.000 muestras).
audio = np.sin(2 * np.pi * 220 * np.arange(16000) / 16000)
# Ráster sintético: 4 bandas de 32x32 píxeles con valores 0-1.
raster = rng.uniform(0, 1, size=(4, 32, 32))

# Resumen de la forma (shape) de cada modalidad.
resumen = pd.DataFrame([
    ["Tabular", tabla.shape, "columnas"],
    ["Texto tokenizado", (len(tokens_didacticos),), "tokens"],
    ["Imagen RGB", imagen.shape, "alto, ancho, canales"],
    ["Audio de 1 segundo", audio.shape, "muestras a 16 kHz"],
    ["Ráster 4 bandas", raster.shape, "bandas, alto, ancho"],
], columns=["modalidad", "forma", "lectura"])
resumen

In [ ]:
# --- Visualización: las 5 modalidades en una sola figura ---
# Cada panel muestra la "forma" de una modalidad distinta.
# Observá cómo cambia la estructura: filas, tokens, píxeles, onda, grilla.

# Creamos una grilla de 2x3 paneles para las 5 modalidades (+1 extra).
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
axes = axes.flatten()

# 1) Tabular: una mini tabla
ax = axes[0]
ax.axis("off")
ax.set_title("1) Tabular\nfilas = casos, columnas = features", fontsize=10)
# Dibujamos la tabla como una grilla de celdas.
tabla_tabla = ax.table(
    cellText=tabla.round(0).values,
    colLabels=tabla.columns,
    loc="center",
    cellLoc="center",
)
tabla_tabla.auto_set_font_size(False)
tabla_tabla.set_fontsize(9)
tabla_tabla.scale(1, 1.6)

# 2) Texto: tokens como cajas
ax = axes[1]
ax.axis("off")
ax.set_title("2) Texto\ntokens en orden", fontsize=10)
# Cada token se dibuja como una caja, respetando su orden.
for i, token in enumerate(tokens_didacticos):
    ax.text(
        i * 1.1, 0, token,
        ha="center", va="center",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="lightblue", edgecolor="steelblue"),
        fontsize=9,
    )
ax.set_xlim(-0.5, len(tokens_didacticos) * 1.1)
ax.set_ylim(-1, 1)

# 3) Imagen RGB: la foto real
ax = axes[2]
ax.imshow(imagen)
ax.set_title("3) Imagen RGB\n(alto, ancho, 3)", fontsize=10)
ax.axis("off")

# 4) Audio: la onda (waveform)
ax = axes[3]
# Graficamos los primeros 400 milisegundos de la onda.
ax.plot(np.arange(400) / 16000, audio[:400], color="darkorange", lw=1)
ax.set_title("4) Audio\namplitud en el tiempo", fontsize=10)
ax.set_xlabel("segundos")
ax.set_ylabel("amplitud")
ax.set_xlim(0, 400 / 16000)

# 5) Ráster: una banda como grilla de colores
ax = axes[4]
ax.imshow(raster[0], cmap="viridis")
ax.set_title("5) Ráster multibanda\nuna banda = una grilla", fontsize=10)
ax.axis("off")

# 6) Ráster: las 4 bandas juntas
ax = axes[5]
# Pegamos las 4 bandas lado a lado para verlas de una vez.
ax.imshow(np.hstack([raster[b] for b in range(4)]), cmap="viridis")
ax.set_title("5b) Ráster: 4 bandas lado a lado", fontsize=10)
ax.axis("off")

plt.suptitle("Cada modalidad tiene una forma distinta de números", fontsize=13)
plt.tight_layout()
plt.show()

## 2. Dato crudo, feature y embedding

Estas tres representaciones corresponden a momentos distintos:

```text
dato crudo -> transformación elegida -> features
                                    |
                                    v
                               modelo encoder
                                    |
                                    v
                                embedding
```

> **Analogía de la receta.** El dato crudo es como los ingredientes
> tal como llegan de la verdulería (tomates, cebolla, ajo). Las
> features son como picarlos y medirlos: "2 tomates, 1 cebolla".
> El embedding es como el plato terminado: ya no reconocés cada
> ingrediente por separado, pero el resultado los resume a todos
> juntos.

| Nivel | Quién o qué lo produce | Ejemplo |
|---|---|---|
| Dato crudo | Fuente o sensor | Píxeles, waveform, texto |
| Feature | Regla de preparación o medición | Edad, energía, NDVI |
| Embedding | Modelo encoder | Vector de 384 o 512 valores |

Una feature puede ser directamente interpretable: “edad = 42”.
Una dimensión de un embedding normalmente no tiene una explicación
aislada del tipo “el valor 17 representa turnos”. La información se
distribuye entre muchas dimensiones.

### Qué hace un encoder

Un encoder transforma una entrada en una representación:

```text
documento -> encoder de texto -> vector
imagen    -> encoder visual   -> vector
audio     -> encoder acústico -> vector
```

Contenidos parecidos para la tarea de entrenamiento suelen quedar
próximos en ese espacio. “Parecido” depende del encoder y de los
datos con los que fue entrenado.

Un embedding no es “el significado verdadero”. Depende del modelo,
sus datos de entrenamiento y la tarea para la que fue ajustado.

### Espacios compatibles e incompatibles

No se comparan directamente:

- embeddings producidos por modelos distintos;
- un embedding de 384 dimensiones con otro de 512;
- vectores de imagen y texto entrenados por separado.

> **Analogía de las unidades de medida.** Comparar embeddings de
> modelos distintos es como sumar metros con libras: son medidas
> válidas, pero no se pueden mezclar. Solo tiene sentido comparar
> vectores que viven en el mismo espacio y fueron producidos por el
> mismo encoder.

Algunos modelos se entrenan para crear un **espacio compartido**.
Por ejemplo, texto e imagen pueden volverse comparables si el
entrenamiento los alineó explícitamente. La compatibilidad es una
propiedad del modelo, no del hecho de que ambos sean vectores.

## 3. Similitud coseno

Una vez que pregunta y documentos están en el mismo espacio,
necesitamos una regla para comparar vectores.

> **Analogía de las flechas.** Imaginá cada vector como una flecha
> que sale del origen. Dos flechas que apuntan casi en la misma
> dirección representan contenidos parecidos. Dos flechas que apuntan
> a lados opuestos representan contenidos muy distintos. La similitud
> coseno mide **cuánto apuntan en la misma dirección**, sin importar
> cuán largas sean las flechas.

La similitud coseno compara su **dirección**, no su tamaño. Podemos
imaginar cada vector como una flecha que parte del origen:

```text
dirección parecida       dirección diferente

     b                           b
    /                            ^
   /                             |        a ---->
  a
```

- cercana a `1`: representaciones alineadas;
- cercana a `0`: poca relación en ese espacio;
- cercana a `-1`: direcciones opuestas.

Normalizar un vector significa ajustar su longitud a 1 sin cambiar
la dirección. Con vectores normalizados, el producto punto produce
la misma comparación que el coseno.

Para dos vectores de tres valores, el producto punto realiza:

```text
[a1, a2, a3] · [b1, b2, b3]
        =
a1*b1 + a2*b2 + a3*b3
```

> **Pensalo así:** el producto punto es como "multiplicar posición
> por posición y sumar todo". Si dos vectores apuntan igual, los
> productos se suman y el resultado es grande. Si apuntan en
> direcciones opuestas, los productos se cancelan y el resultado
> es chico.

### Qué no significa un score alto

Una similitud alta no verifica:

- que un documento sea verdadero;
- que contenga la respuesta completa;
- que dos casos sean equivalentes;
- que exista una relación causal;
- que el resultado sea seguro para tomar una decisión.

El score solo expresa cercanía dentro del espacio aprendido.

In [ ]:
# --- Funciones base: normalizar y similitud coseno ---
# Normalizar: ajusta la longitud del vector a 1 sin cambiar su dirección.
def normalizar(vector):
    vector = np.asarray(vector, dtype=float)
    norma = np.linalg.norm(vector)
    return vector / norma if norma else vector

# Similitud coseno: producto punto entre dos vectores normalizados.
def similitud_coseno(a, b):
    return float(normalizar(a) @ normalizar(b))

# Tres vectores de ejemplo en el mismo espacio (3 dimensiones).
vectores = {
    "consulta_turno": [0.9, 0.8, 0.1],
    "agenda_medica": [0.8, 0.9, 0.1],
    "imagen_satelital": [0.1, 0.0, 0.95],
}
# Comparamos por pares: ¿cuánto apuntan en la misma dirección?
pd.DataFrame([
    [a, b, similitud_coseno(vectores[a], vectores[b])]
    for a, b in [
        ("consulta_turno", "agenda_medica"),
        ("consulta_turno", "imagen_satelital"),
    ]
], columns=["vector_a", "vector_b", "similitud"]).round(3)

In [ ]:
# --- Visualización: vectores como flechas en 2D ---
# Reducimos cada vector a 2 dimensiones para poder dibujarlo.
# Fijate cómo "consulta_turno" y "agenda_medica" apuntan casi igual,
# mientras que "imagen_satelital" apunta en otra dirección.
#
# Ojo: al quedar solo 2 dimensiones, los scores cambian respecto de
# la tabla de 3 dimensiones de arriba. Eso es esperable: cada versión
# del vector vive en un espacio distinto. Lo importante aquí es ver
# la dirección de las flechas, no comparar números entre tablas.

# Versión 2D de los mismos vectores, solo para poder dibujarlos.
vectores_2d = {
    "consulta_turno": [0.9, 0.8],
    "agenda_medica": [0.8, 0.9],
    "imagen_satelital": [0.1, 0.0],
}

# Creamos el lienzo y asignamos un color a cada vector.
fig, ax = plt.subplots(figsize=(6, 6))
colores = {
    "consulta_turno": "tab:blue",
    "agenda_medica": "tab:green",
    "imagen_satelital": "tab:red",
}
# Dibujamos cada vector como una flecha desde el origen (0,0).
for nombre, (x, y) in vectores_2d.items():
    ax.arrow(
        0, 0, x, y,
        head_width=0.06, head_length=0.06,
        fc=colores[nombre], ec=colores[nombre],
        length_includes_head=True,
    )
    # Etiquetamos la punta de cada flecha con su nombre.
    ax.text(x * 1.15, y * 1.15, nombre, color=colores[nombre], fontsize=9)

# Ajustamos los límites y agregamos ejes de referencia.
ax.set_xlim(-0.2, 1.3)
ax.set_ylim(-0.2, 1.3)
ax.axhline(0, color="gray", lw=0.8)
ax.axvline(0, color="gray", lw=0.8)
ax.set_aspect("equal")
ax.set_title(
    "Vectores como flechas desde el origen\n"
    "(2 dimensiones para poder dibujarlos)"
)
ax.set_xlabel("dimensión 1")
ax.set_ylabel("dimensión 2")
plt.grid(alpha=0.3)
plt.show()

# Mostramos el ángulo entre los vectores (en grados)
# En 2D podemos "ver" el ángulo; en más dimensiones no se puede
# dibujar, pero el coseno lo calcula igual.
def angulo_grados(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    cos = similitud_coseno(a, b)
    return np.degrees(np.arccos(np.clip(cos, -1, 1)))

# Tabla final: similitud y ángulo entre cada par de vectores.
pd.DataFrame([
    [a, b,
     round(similitud_coseno(vectores_2d[a], vectores_2d[b]), 3),
     round(angulo_grados(vectores_2d[a], vectores_2d[b]), 1)]
    for a, b in [
        ("consulta_turno", "agenda_medica"),
        ("consulta_turno", "imagen_satelital"),
    ]
], columns=["vector_a", "vector_b", "similitud", "ángulo (grados)"])

---
## 4. Experimento: búsqueda semántica de texto

Una búsqueda por palabras encuentra coincidencias literales. Una
búsqueda semántica intenta recuperar contenido relacionado aunque
la pregunta use otras palabras.

> **Analogía del bibliotecario.** Una búsqueda por palabras es como
> pedirle a un bibliotecario que busque el libro que contiene la
> palabra exacta "cita". Una búsqueda semántica es como pedirle que
> entienda *qué querés hacer* ("quiero cambiar mi turno") y te traiga
> los libros que hablan de eso, aunque usen otras palabras.

```text
pregunta: "¿puedo cambiar la fecha de mi cita?"
documento: "los turnos pueden reprogramarse..."

coincidencia literal: pocas palabras iguales
relación semántica:   intención parecida
```

El pipeline tiene dos momentos.

**Preparación de documentos:**

```text
documentos -> encoder -> embeddings -> almacenamiento
```

**Búsqueda:**

```text
pregunta -> mismo encoder -> embedding
                             |
                             v
              similitud con documentos
                             |
                             v
                      ranking top-k
```

`top-k` significa conservar los `k` resultados con mayor score.
Recuperar no es generar: el resultado de esta etapa son documentos
o fragmentos ordenados.

> **Pensalo así:** el ranking es como una lista de "más parecido a
> menos parecido". El primer lugar es el que el modelo cree que mejor
> responde a la pregunta. Pero el modelo puede equivocarse, por eso
> siempre hay que revisar el resultado.



```bash
pregunta = "¿Con cuánta anticipación puedo cambiar un turno?"
```

In [ ]:
# --- Búsqueda semántica: documentos y pregunta ---
# Cuatro documentos de ejemplo que harán de "biblioteca".
documentos = pd.DataFrame([
    ["turnos", "Los turnos pueden reprogramarse hasta 24 horas antes."],
    ["sensores", "La calibración de sensores se realiza cada seis meses."],
    ["recetas", "Las recetas crónicas requieren validación profesional."],
    ["satelite", "Las bandas roja e infrarroja permiten calcular NDVI."],
], columns=["id", "texto"])

# La pregunta que queremos responder con la búsqueda.
pregunta = "¿Con cuánta anticipación puedo cambiar un turno?"

# La práctica intenta usar un encoder preentrenado. Si no está
# disponible, usa TF-IDF como fallback. TF-IDF representa términos y
# frecuencia; no tiene la misma capacidad semántica, pero permite
# mantener visible el contraste entre representaciones.
try:
    from sentence_transformers import SentenceTransformer

    # Cargamos el encoder de texto preentrenado (all-MiniLM-L6-v2).
    encoder_texto = SentenceTransformer(
        "sentence-transformers/all-MiniLM-L6-v2", device="cpu"
    )
    # Convertimos cada documento en un embedding normalizado.
    emb_documentos = encoder_texto.encode(
        documentos["texto"].tolist(),
        normalize_embeddings=True,
        show_progress_bar=True,
    )
    # Convertimos la pregunta en un embedding con el mismo encoder.
    emb_pregunta = encoder_texto.encode(
        [pregunta],
        normalize_embeddings=True,
        show_progress_bar=True,
    )[0]
    modo_texto = "embeddings all-MiniLM-L6-v2"
except Exception as error:
    from sklearn.feature_extraction.text import TfidfVectorizer

    # Fallback: representamos documentos y pregunta con TF-IDF.
    vectorizador = TfidfVectorizer(
        ngram_range=(1, 2), norm="l2"
    )
    # La matriz incluye los documentos y la pregunta como última fila.
    matriz = np.asarray(vectorizador.fit_transform(
        documentos["texto"].tolist() + [pregunta]
    ).todense())
    emb_documentos = matriz[:-1]
    emb_pregunta = matriz[-1]
    modo_texto = (
        "fallback TF-IDF local "
        f"({type(error).__name__})"
    )

# Similitud coseno entre la pregunta y cada documento (producto punto).
scores = np.asarray(emb_documentos @ emb_pregunta).flatten()
# Armamos el ranking: copiamos los documentos y agregamos el score.
ranking = documentos.copy()
ranking["similitud"] = scores
print("Modo:", modo_texto)
# Ordenamos de mayor a menor similitud (el más parecido primero).
ranking.sort_values("similitud", ascending=False).round(3)

In [ ]:
# --- Visualización: el ranking como barras ---
# Cada barra es un documento. La más larga es la que el modelo
# considera más parecida a la pregunta.

# Ordenamos de menor a mayor para que el primero quede arriba.
ranking_ordenado = ranking.sort_values("similitud", ascending=True)

# Creamos el gráfico de barras horizontales.
fig, ax = plt.subplots(figsize=(9, 4))
barras = ax.barh(
    ranking_ordenado["id"],
    ranking_ordenado["similitud"],
    color="steelblue",
)
# Resaltamos el primer lugar con otro color.
barras[-1].set_color("tab:orange")
ax.set_xlabel("similitud con la pregunta")
ax.set_title(f"Ranking de documentos — {modo_texto}")
ax.set_xlim(0, 1)
# Escribimos el valor exacto al final de cada barra.
for i, (_, fila) in enumerate(ranking_ordenado.iterrows()):
    ax.text(fila["similitud"] + 0.01, i, f"{fila['similitud']:.3f}",
            va="center", fontsize=9)
plt.tight_layout()
plt.show()

### Qué observar

Leé el ranking antes de mirar solamente el score:

1. ¿El fragmento correcto aparece primero?
2. ¿Aparece dentro de los primeros `k` resultados?
3. ¿Los fragmentos recuperados contienen evidencia suficiente?
4. ¿Hay resultados cercanos pero irrelevantes?

La pregunta y el fragmento no necesitan usar exactamente las mismas
palabras. Aun así, el encoder puede acercarlos.

Retrieval puede equivocarse. Por eso se evalúa de forma separada a
cualquier LLM que use después los fragmentos. Si la evidencia
correcta no fue recuperada, el generador no la recibirá.

Métricas como `Recall@k` preguntan si el resultado relevante apareció
dentro de los primeros `k`. En esta clase hacemos la inspección de
forma manual porque el conjunto es pequeño.

---
## 5. Embeddings de imagen

Un clasificador visual puede separarse conceptualmente en dos partes:

```text
imagen
   |
   v
extractor de features -> embedding -> head de clasificación -> clases
```

> **Analogía del ojo experto.** El extractor de features es como un
> ojo experto que mira la imagen y anota "qué hay y cómo se ve"
> (colores, texturas, formas). Ese resumen es el embedding. El head
> es como la persona que lee esas notas y dice "esto es una flor".
> Si solo queremos comparar imágenes, nos quedamos con las notas
> (el embedding) y no necesitamos la etiqueta final.

ResNet18 fue entrenada para clasificación. Antes de la última capa
produce un vector de features visuales. El **head** es la capa final
que transforma ese vector en clases. Al retirar el head y
conservar el extractor, podemos reutilizar ese vector como
embedding.

La comparación visual puede responder preguntas como:

- ¿qué imagen de una galería se parece más a esta consulta?;
- ¿hay duplicados o variantes casi iguales?;
- ¿qué ejemplos debería revisar una persona?;

El concepto de parecido puede incluir color, textura, composición y
objetos. No necesariamente coincide con el criterio del negocio.

En la galería esperamos que la flor espejada y la flor con menos
color queden más cerca de la flor original que el paisaje. Si no hay
pesos preentrenados disponibles, el fallback usa histogramas RGB:
compara color, no semántica profunda.

In [ ]:
# --- Embeddings de imagen con ResNet18 ---
# Importamos PyTorch y las herramientas de imagen.
import torch
import torch.nn.functional as F
from PIL import Image, ImageEnhance, ImageOps
from torchvision.models import ResNet18_Weights, resnet18

# Limitamos los hilos de CPU para no saturar la máquina.
torch.set_num_threads(min(4, torch.get_num_threads()))
# Cargamos dos fotos de ejemplo y armamos la galería con variantes.
flor = Image.fromarray(load_sample_image("flower.jpg")).convert("RGB")
paisaje = Image.fromarray(load_sample_image("china.jpg")).convert("RGB")
galeria = {
    "flor": flor,
    "flor_espejo": ImageOps.mirror(flor),
    "flor_sin_color": ImageEnhance.Color(flor).enhance(0.2),
    "paisaje": paisaje,
}

# Intentamos cargar ResNet18 preentrenada. Si no hay pesos disponibles,
# usamos un fallback con histogramas RGB (compara color, no semántica).
try:
    weights = ResNet18_Weights.DEFAULT
    modelo_imagen = resnet18(weights=weights)
    # Quitamos la última capa (head) para quedarnos con el extractor.
    extractor = torch.nn.Sequential(
        *list(modelo_imagen.children())[:-1]
    )
    extractor.eval()
    # Preprocesamiento estándar que espera la red (tamaño, normalización).
    preprocesar = weights.transforms()
    modo_imagen = "features ResNet18 preentrenada"
except Exception as error:
    extractor = None
    preprocesar = None
    modo_imagen = (
        "fallback histograma RGB local "
        f"({type(error).__name__})"
    )

# Función que convierte una imagen en su embedding (vector normalizado).
def embedding_imagen(img):
    if extractor is not None:
        # Preprocesamos, agregamos la dimensión de batch y pasamos por la red.
        lote = preprocesar(img).unsqueeze(0)
        with torch.no_grad():
            vector = extractor(lote).flatten(1)
        return F.normalize(vector, dim=1)[0]

    # Fallback: histograma de color por canal RGB (48 valores).
    array = np.asarray(img.resize((64, 64)), dtype=np.float32) / 255
    histograma = np.concatenate([
        np.histogram(
            array[..., canal],
            bins=16,
            range=(0, 1),
            density=True,
        )[0]
        for canal in range(3)
    ])
    vector = torch.tensor(histograma, dtype=torch.float32)
    return F.normalize(vector, dim=0)

# Calculamos el embedding de cada imagen de la galería.
emb_imagenes = {k: embedding_imagen(v) for k, v in galeria.items()}
# Mostramos la galería completa en una fila de paneles.
fig, axes = plt.subplots(1, len(galeria), figsize=(14, 3))
for ax, (nombre, imagen_galeria) in zip(axes, galeria.items()):
    ax.imshow(imagen_galeria)
    ax.set_title(nombre)
    ax.axis("off")
plt.suptitle(f"Galería comparada — {modo_imagen}")
plt.tight_layout()
plt.show()

# Elegimos la consulta y comparamos su embedding con el resto.
consulta = "flor"
similitudes = pd.DataFrame([
    [nombre, float(emb_imagenes[consulta] @ vector)]
    for nombre, vector in emb_imagenes.items()
    if nombre != consulta
], columns=["imagen", "similitud"])
# Ordenamos de mayor a menor: la más parecida primero.
similitudes.sort_values("similitud", ascending=False).round(3)

In [ ]:
# --- Visualización: similitudes de imagen como barras ---
# La barra más larga es la imagen que el modelo considera más parecida
# a la consulta ("flor"). Esperamos que las variantes de la flor
# superen al paisaje.

# Ordenamos de menor a mayor para que la más parecida quede arriba.
similitudes_ordenadas = similitudes.sort_values("similitud", ascending=True)

# Creamos el gráfico de barras horizontales.
fig, ax = plt.subplots(figsize=(8, 3.5))
barras = ax.barh(
    similitudes_ordenadas["imagen"],
    similitudes_ordenadas["similitud"],
    color="mediumseagreen",
)
# Resaltamos el primer lugar con otro color.
barras[-1].set_color("tab:orange")
ax.set_xlabel("similitud con la consulta 'flor'")
ax.set_title(f"¿Qué imagen se parece más a la flor? — {modo_imagen}")
ax.set_xlim(0, 1)
# Escribimos el valor exacto al final de cada barra.
for i, (_, fila) in enumerate(similitudes_ordenadas.iterrows()):
    ax.text(fila["similitud"] + 0.01, i, f"{fila['similitud']:.3f}",
            va="center", fontsize=9)
plt.tight_layout()
plt.show()

## 6. Una representación preparada de audio

Un encoder de audio puede resumir una señal completa o un fragmento:

```text
waveform -> preprocesamiento acústico -> encoder -> embedding
```

> **Analogía de la huella de voz.** El waveform es la "huella" de la
> onda de sonido: sube y baja con la amplitud a lo largo del tiempo.
> El encoder la resume en un embedding, como si convirtiéramos una
> melodía completa en una "firma" corta. Dos firmas parecidas
> significan sonidos parecidos para ese modelo.

Según su entrenamiento, el embedding puede representar:

- contenido hablado;
- identidad o características de voz;
- tipo de sonido;
- ambiente acústico;
- emoción aparente.

Es importante conocer el objetivo del modelo antes de interpretar
cercanía.

Para no descargar otro encoder en esta clase usamos vectores
didácticos preparados. El “original” y su versión con ruido se
construyen deliberadamente próximos; “alarma” se genera por separado.
Estos vectores **no fueron extraídos de un audio real** y no permiten
evaluar un modelo acústico.

En la clase 6 veremos waveform, espectrograma y Whisper sobre un WAV.

In [ ]:
# --- Embeddings de audio didácticos ---
# Generamos un vector base aleatorio que hará de "audio original".
base_audio = normalizar(rng.normal(size=12))
# Creamos tres "audios": el original, el mismo con ruido y una alarma.
embeddings_audio = {
    "consulta_original": base_audio,
    "consulta_con_ruido": normalizar(
        base_audio + rng.normal(0, 0.05, size=12)
    ),
    "alarma": normalizar(rng.normal(size=12)),
}
# Comparamos cada audio con el original usando similitud coseno.
pd.DataFrame([
    [nombre, similitud_coseno(
        embeddings_audio["consulta_original"], vector
    )]
    for nombre, vector in embeddings_audio.items()
    if nombre != "consulta_original"
], columns=["audio", "similitud"]).round(3)

In [ ]:
# --- Visualización: similitudes de audio como barras ---
# La versión con ruido debería quedar más cerca del original que la
# alarma, porque representa "el mismo sonido con algo de interferencia".

# Recalculamos las similitudes en un DataFrame para graficarlas.
similitudes_audio = pd.DataFrame([
    [nombre, similitud_coseno(
        embeddings_audio["consulta_original"], vector
    )]
    for nombre, vector in embeddings_audio.items()
    if nombre != "consulta_original"
], columns=["audio", "similitud"])

# Ordenamos de menor a mayor para que la más parecida quede arriba.
similitudes_audio_ordenadas = similitudes_audio.sort_values("similitud", ascending=True)

# Creamos el gráfico de barras horizontales.
fig, ax = plt.subplots(figsize=(8, 3))
barras = ax.barh(
    similitudes_audio_ordenadas["audio"],
    similitudes_audio_ordenadas["similitud"],
    color="darkorange",
)
# Resaltamos el primer lugar con otro color.
barras[-1].set_color("tab:orange")
ax.set_xlabel("similitud con la consulta original")
ax.set_title("¿Qué audio se parece más al original?")
ax.set_xlim(0, 1)
# Escribimos el valor exacto al final de cada barra.
for i, (_, fila) in enumerate(similitudes_audio_ordenadas.iterrows()):
    ax.text(fila["similitud"] + 0.01, i, f"{fila['similitud']:.3f}",
            va="center", fontsize=9)
plt.tight_layout()
plt.show()

## Actividad — elegir representación y comparación

Para cada caso separá cuatro decisiones:

| Decisión | Pregunta |
|---|---|
| Modalidad | ¿Qué tipo de dato llega al sistema? |
| Representación | ¿Qué estructura debe conservarse? |
| Modelo | ¿Qué encoder o algoritmo la procesará? |
| Comparación/salida | ¿Qué necesita consumir la aplicación? |

> **Cómo pensarlo en 3 pasos:**
> 1. **Mirá la entrada:** ¿qué llega? ¿una tabla, un texto, una foto,
>    un audio, un mapa?
> 2. **Pensá qué hay que conservar:** ¿el orden de las palabras? ¿la
>    posición de cada píxel? ¿la ubicación geográfica?
> 3. **Elegí cómo comparar:** ¿vas a medir cercanía entre vectores,
>    buscar coincidencias de texto o superponer mapas?

Modificá una fila o agregá un caso. Evitá comparar embeddings de
espacios incompatibles. Si proponés un espacio compartido, indicá
qué modelo fue entrenado para alinear las modalidades.

In [ ]:
# --- Actividad: decisiones de representación ---
# Cada fila plantea un problema y su representación elegida.
decisiones = pd.DataFrame([
    ["Buscar una política similar a una pregunta", "embedding de texto"],
    ["Encontrar imágenes visualmente cercanas", "embedding de imagen"],
    ["Medir cobertura en cada píxel", "ráster/máscara"],
    ["Transcribir una llamada", "waveform + modelo ASR"],
    ["Predecir demanda desde columnas", "features tabulares"],
], columns=["problema", "representacion"])

# TODO: agregá un caso de tu industria y justificá la representación.
# Agregamos la justificación de cada elección.
decisiones["por_que"] = [
    "Pregunta y documentos deben compartir un espacio semántico.",
    "Se comparan representaciones del mismo encoder visual.",
    "La salida conserva una posición geográfica por celda.",
    "ASR recibe amplitud muestreada en el tiempo.",
    "Cada fila contiene las variables de un caso.",
]
decisiones

---

## Síntesis de la clase

- Cada modalidad tiene una forma y un preprocesamiento propios.
- Features y embeddings no son sinónimos.
- La similitud coseno compara vectores dentro de un mismo espacio.
- Embeddings permiten recuperación y reutilización de representaciones.
- Una similitud alta no garantiza una respuesta correcta.

## Comprobación conceptual

Antes de continuar, intentá responder sin mirar el notebook:

1. ¿Cuál era el problema central de la clase?
2. ¿Qué entrada recibió el sistema y qué salida produjo?
3. ¿Qué decisión humana siguió siendo necesaria?
4. ¿Qué limitación observaste en el experimento?

Si podés explicarlo con tus propias palabras y justificarlo con un resultado visible, alcanzaste el objetivo introductorio.

## Puente con la próxima clase

La clase 4 muestra cómo las redes aprenden esas representaciones y compara PyTorch con Keras.

## Conexión con los tracks

Salud usará embeddings para NLP, audio y RAG; Imagen los reutilizará para clasificación, detección y análisis multibanda.

La implementación profunda, el trabajo con datasets reales y las decisiones de producción se desarrollarán en los módulos especializados.